In [1]:
from pathlib import Path

BASE = Path.home() / "Documents/GalFields/Tools"
BASE.mkdir(parents=True, exist_ok=True)

REGION_DIR = BASE / "regions"
CUTOUT_DIR = BASE / "cutouts"
PLOT_DIR   = BASE / "plots"

REGION_DIR.mkdir(parents=True, exist_ok=True)
CUTOUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("Working directory:", BASE)

Working directory: /Users/macarias/Documents/GalFields/Tools


# Imports and Functions

In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u

from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.stats import sigma_clipped_stats
from astropy.convolution import convolve_fft

from regions import Regions
from radio_beam import Beam

from reproject import reproject_interp
from reproject.mosaicking import reproject_and_coadd
from reproject.adaptive import reproject_adaptive

import aplpy

In [2]:
def parse_region(regfile):
    reg = Regions.read(regfile, format="ds9")[0]
    return (
        reg.center,
        reg.width.to(u.deg).value,
        reg.height.to(u.deg).value
    )


In [3]:
def overlaps(path, center, wdeg, hdeg):

    with fits.open(path) as hdul:
        w = WCS(hdul[0].header)
        ny, nx = hdul[0].data.shape

    dx = (wdeg/2) * u.deg
    dy = (hdeg/2) * u.deg

    corners = SkyCoord(
        l=[center.l-dx, center.l+dx, center.l-dx, center.l+dx],
        b=[center.b-dy, center.b-dy, center.b+dy, center.b+dy],
        frame="galactic"
    )

    xpix, ypix = w.world_to_pixel(corners)

    return np.any(
        (xpix >= 0) & (xpix < nx) &
        (ypix >= 0) & (ypix < ny)
    )


In [4]:
def build_cutout(paths, center, wdeg, hdeg, outpath):

    with fits.open(paths[0]) as hdul:
        w_in = WCS(hdul[0].header)
        pix_deg = float(np.mean(proj_plane_pixel_scales(w_in)))

    nx = int(wdeg / pix_deg)
    ny = int(hdeg / pix_deg)

    w_out = WCS(naxis=2)
    w_out.wcs.crpix = [nx/2, ny/2]
    w_out.wcs.cdelt = [-pix_deg, pix_deg]
    w_out.wcs.crval = [center.l.deg, center.b.deg]
    w_out.wcs.ctype = ["GLON-TAN","GLAT-TAN"]

    arrays, weights = [], []

    for p in paths:
        with fits.open(p) as hdul:
            arr, fp = reproject_interp(
                (hdul[0].data.squeeze(), WCS(hdul[0].header)),
                w_out,
                shape_out=(ny,nx)
            )
        arrays.append(arr)
        weights.append(fp)

    mosaic = np.nansum([a*w for a,w in zip(arrays,weights)], axis=0)
    mosaic /= np.nansum(weights, axis=0)

    # Copy beam from first input file
    with fits.open(paths[0]) as hdul:
        hdr_in = hdul[0].header

    hdr_out = w_out.to_header()

    for key in ["BMAJ", "BMIN", "BPA"]:
        if key in hdr_in:
            hdr_out[key] = hdr_in[key]

    fits.writeto(outpath, mosaic, hdr_out, overwrite=True)


    return outpath


In [5]:
def align_and_compute_spx(lofar_fits, mk_fits, out_spx):

    with fits.open(lofar_fits) as h:
        lf = h[0].data.squeeze().astype(float)
        w_lf = WCS(h[0].header)
        beam_lf = Beam.from_fits_header(h[0].header)

    with fits.open(mk_fits) as h:
        mk = h[0].data.squeeze().astype(float)
        w_mk = WCS(h[0].header)
        beam_mk = Beam.from_fits_header(h[0].header)

    # Choose coarser beam as reference
    if beam_mk.sr >= beam_lf.sr:
        ref_data, ref_wcs, ref_beam = mk, w_mk, beam_mk
        other, other_wcs, other_beam = lf, w_lf, beam_lf
    else:
        ref_data, ref_wcs, ref_beam = lf, w_lf, beam_lf
        other, other_wcs, other_beam = mk, w_mk, beam_mk

    # Smooth other → ref beam
    try:
        diff = ref_beam.deconvolve(other_beam)
        pix = np.mean(np.abs(other_wcs.wcs.cdelt)) * u.deg
        kernel = diff.as_kernel(pix)
        other = convolve_fft(other, kernel, allow_huge=True)
    except:
        pass

    # Flux-safe reprojection
    def pix_area_sr(w):
        cd = np.abs(w.wcs.cdelt)
        return cd[0]*cd[1]*(np.pi/180)**2

    beam_in = other_beam.sr.value
    beam_out = ref_beam.sr.value

    other_jypix = other * (pix_area_sr(other_wcs) / beam_in)

    other_on_ref_pix, _ = reproject_and_coadd(
        [(other_jypix, other_wcs)],
        ref_wcs,
        shape_out=ref_data.shape,
        reproject_function=reproject_adaptive,
        conserve_flux=True
    )

    other_on_ref = other_on_ref_pix * (beam_out / pix_area_sr(ref_wcs))

    # Compute SPX
    alpha = np.log(other_on_ref / ref_data) / np.log(MEERKAT_FREQ_HZ / LOFAR_FREQ_HZ)

    fits.writeto(out_spx, alpha, ref_wcs.to_header(), overwrite=True)

    return out_spx, ref_data


In [6]:
def plot_tripanel(lofar, mk, spx, ref_data, title, outpath):

    lofar = str(lofar)
    mk = str(mk)
    spx = str(spx)

    mk_data = ref_data
    spx_data = fits.getdata(spx)

    _,_,noise = sigma_clipped_stats(mk_data)
    mask = mk_data > SIGMA_FACTOR * noise
    spx_masked = np.where(mask, spx_data, np.nan)

    masked = Path(spx).parent / "spx_masked.fits"
    fits.writeto(masked, spx_masked, fits.getheader(spx), overwrite=True)

    fig = plt.figure(figsize=(14,5))
    fig.suptitle(title)

    f1 = aplpy.FITSFigure(lofar, figure=fig, subplot=(1,3,1))
    f1.show_colorscale(cmap="Blues", stretch="sqrt")

    f2 = aplpy.FITSFigure(mk, figure=fig, subplot=(1,3,2))
    f2.show_colorscale(cmap="Blues", stretch="sqrt")

    f3 = aplpy.FITSFigure(str(masked), figure=fig, subplot=(1,3,3))
    f3.show_colorscale(cmap=SPX_CMAP, vmin=SPX_VMIN, vmax=SPX_VMAX)
    f3.add_colorbar()

    plt.tight_layout()
    plt.savefig(outpath)
    plt.close(fig)


# Config and Runner

In [7]:
REGION_DIR = Path("./regions")
CUTOUT_DIR = Path("./cutouts")
PLOT_DIR   = Path("./plots")

MEERKAT_ROOT = Path("/Users/macarias/Documents/GalFields/MGPS")
LOFAR_ROOT   = Path("/Users/macarias/Documents/GalFields/Mosaics")

USE_LOFAR_LOWRES = True

LOFAR_FREQ_HZ   = 144e6
MEERKAT_FREQ_HZ = 1284e6

SIGMA_FACTOR = 3.5

SPX_VMIN = -1
SPX_VMAX = 1
SPX_CMAP = "coolwarm"

CUTOUT_DIR.mkdir(exist_ok=True)
PLOT_DIR.mkdir(exist_ok=True)

In [8]:
# ===================== RUNNER =====================

for regfile in sorted(REGION_DIR.glob("*.reg")):

    region_name = regfile.stem
    print(f"\nProcessing {region_name}")

    outdir = CUTOUT_DIR / region_name
    outdir.mkdir(exist_ok=True)

    center, wdeg, hdeg = parse_region(regfile)

    mk_paths = [p for p in MEERKAT_ROOT.glob("*.fits")
                if overlaps(p, center, wdeg, hdeg)]

    lf_paths = []
    for d in LOFAR_ROOT.iterdir():
        fname = "low-mosaic-blanked.fits" if USE_LOFAR_LOWRES else "mosaic-blanked.fits"
        f = d/fname
        if f.exists() and overlaps(f, center, wdeg, hdeg):
            lf_paths.append(f)

    mk_cut = build_cutout(mk_paths, center, wdeg, hdeg,
                          outdir/"meerkat_cutout.fits")

    lf_cut = build_cutout(lf_paths, center, wdeg, hdeg,
                          outdir/"lofar_cutout.fits")

    spx, ref_data = align_and_compute_spx(
        lf_cut, mk_cut,
        outdir/"spectral_index.fits"
    )

    plot_tripanel(
        lf_cut, mk_cut, spx,
        ref_data,
        region_name,
        PLOT_DIR/f"{region_name}.pdf"
    )

print("\nAll regions complete.")



Processing 1LHAASO_J1857+0245_widest


/var/folders/s3/1q4jkpfd3mqd43nbhh0s2jwc0000gp/T/ipykernel_24728/446534472.py:51: RuntimeWarning: divide by zero encountered in log
  alpha = np.log(other_on_ref / ref_data) / np.log(MEERKAT_FREQ_HZ / LOFAR_FREQ_HZ)
/var/folders/s3/1q4jkpfd3mqd43nbhh0s2jwc0000gp/T/ipykernel_24728/446534472.py:51: RuntimeWarning: invalid value encountered in log
  alpha = np.log(other_on_ref / ref_data) / np.log(MEERKAT_FREQ_HZ / LOFAR_FREQ_HZ)


INFO: Auto-setting vmin to -1.144e-02 [aplpy.core]
INFO: Auto-setting vmax to  1.484e-02 [aplpy.core]


INFO: Auto-setting vmin to -3.219e-03 [aplpy.core]
INFO: Auto-setting vmax to  3.500e-03 [aplpy.core]



All regions complete.
